# Install dependencies

In [1]:
from dotenv import load_dotenv
import os
load_dotenv('var.env')
hf_token = os.getenv("HF_TOKEN")

In [2]:
!pip -q install transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 31.1 MB/s eta 0:00:00


# Create training dataset

In [3]:
from datasets import Dataset

data = [
    {
        "instruction": "What is CAN bus?",
        "output": "CAN bus is a vehicle communication protocol..."
    },
    {
        "instruction": "Explain telematics anomaly detection",
        "output": "Telematics anomaly detection identifies unusual vehicle patterns..."
    },
    {
        "instruction": "What is VIN?",
        "output": "Vehicle Identification Number is a unique identifier..."
    }
]

dataset = Dataset.from_list(data)

# Convert dataset to chat format

In [4]:
def format_dataset(record):
  return {
      "text": f"### Instruction:\n{record['instruction']}\n\n### Response:\n{record['output']}"
  }

dataset = dataset.map(format_dataset)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

# Load model with quantization

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 1. Define the quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16", # Common for T4 GPUs
    bnb_4bit_quant_type="nf4"         # Recommended for better precision
)

model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map='auto',
    token=hf_token
)


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

# Finetune the model - Applying LoRA

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [7]:
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 7,254,839,296 || trainable%: 0.0939


Only 0.09% of trainable parameters

# Tokenize dataset

In [8]:
tokenizer.pad_token = tokenizer.eos_token

In [9]:
def tokenize(example):
    outputs = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    # The Trainer looks for this 'labels' key to calculate loss
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

dataset = dataset.map(tokenize)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

# Training

In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    output_dir="finetuned_model"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Step,Training Loss


TrainOutput(global_step=3, training_loss=9.406253178914389, metrics={'train_runtime': 12.932, 'train_samples_per_second': 0.696, 'train_steps_per_second': 0.232, 'total_flos': 196870945112064.0, 'train_loss': 9.406253178914389, 'epoch': 3.0})

# Save finetuned model

In [12]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

('lora_adapter/tokenizer_config.json',
 'lora_adapter/chat_template.jinja',
 'lora_adapter/tokenizer.json')

# Inference

In [13]:
prompt = "what is VIN"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0]))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> what is VIN number?

A VIN number, or Vehicle Identification Number, is a unique code that is assigned to every vehicle manufactured in the world. It is a 17-digit code that contains information about the vehicle's manufacturer, model, year, engine type, and other important details. The VIN number is used to identify the vehicle in various databases, such as insurance databases, vehicle registration databases, and recall databases. It is also used to track
